# Jobs: latent dim (K)

project = ```_IterativeVAE```, host = ```mach```, device = ```any```

**Motivation**: <br>


In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, '_IterativeVAE'))
from figures.convergence import plot_convergence
from figures.imgs import plot_weights
from figures.fighelper import *
from main.train import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from base.helper import job_runner_script


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = 'Dropbox/git/_IterativeVAE/scripts'
save_dir = pjoin(os.environ['HOME'], save_dir)
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

[
    'cleanup_chkpts.sh',
    'cleanup_recursive.sh',
    'copyfits.sh',
    'fit_model.sh',
    'kill_screens.sh',
    'resume_fit.sh',
    'run_sessions.sh',
    'test_tqdm.py',
    'test_tqdm.sh'
]

## mach

This is the best Poisson model from the main figs:

```bash
./fit_model.sh 0 'vH16-wht' 'poisson' --t_train 16 --n_latents 512 --beta_outer 24.0
```

Now we're gonna explore dependence on the latent dim.

In [4]:
host = 'mach'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
model_type = 'poisson'
dataset = 'vH16-wht'
t_train = 16
beta = 24.0

latent_dims = [32, 1024, 128, 256, 512, 768, 64, 2048]

In [6]:
for k in latent_dims:
    # get arg
    arg = ' '.join([
        f"--t_train {t_train}",
        f"--beta_outer {beta}",
        f"--n_latents '{k}'",
        '--comment rebuttal',
        '--verbose',
        # '--dry_run',
    ])
    gpu_i = tot % 2  # first two gpus
    scripts[gpu_i].append(job_runner_script(
        device=gpu_i,
        dataset=dataset,
        model=model_type,
        args=arg,
        seed=0,
    ))
    tot += 1

In [7]:
print(tot)

8

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 4, 1: 4}

### Save

In [9]:
n_fits = 4

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'mach-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '0' 'vH16-wht' 'poisson' --seed 0 --t_train 16 --beta_outer 24.0 --n_latents '32' --comment rebuttal
--verbose

[PROGRESS] 'mach-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '0' 'vH16-wht' 'poisson' --seed 0 --t_train 16 --beta_outer 24.0 --n_latents '128' --comment 
rebuttal --verbose

[PROGRESS] 'mach-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

...

./fit_model.sh '1' 'vH16-wht' 'poisson' --seed 0 --t_train 16 --beta_outer 24.0 --n_latents '256' --comment 
rebuttal --verbose

[PROGRESS] 'mach-cuda1-fit2.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '1' 'vH16-wht' 'poisson' --seed 0 --t_train 16 --beta_outer 24.0 --n_latents '768' --comment 
rebuttal --verbose

[PROGRESS] 'mach-cuda1-fit3.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '1' 'vH16-wht' 'poisson' --seed 0 --t_train 16 --beta_outer 24.0 --n_latents '2048' --comment 
rebuttal --verbose

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_model.sh '1' 'vH16-wht' 'poisson' --seed 0 --t_train 16 --beta_outer 24.0 --n_latents '2048' --comment 
rebuttal --verbose